# Free Hit - expected points & squad optimisation

Weekly workflow. Run top to bottom; the only cell you normally *edit* is the
minutes-override step.

The model prices every FPL scoring event it can from **live exchange odds** and
falls back to a statistical model where the market cannot reach. Every number
carries a provenance tag so you can see which is which.

## 1. Setup

In [ ]:
# Pick up edits to the fplfh package without restarting the kernel. Without
# this, a running kernel keeps the version of a module it first imported, and
# any function added since fails to import until you restart.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass          # not running under IPython

import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "fplfh").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from fplfh.pipeline import run
from fplfh.optimise import optimise_free_hit, top_squads
from fplfh.scoring import validate_against_api
from fplfh.fpl import FPLClient
from fplfh.model import COMPONENTS

## 2. Confirm the scoring rules

The points-per-event table is read live from the FPL API, but the *thresholds*
(DefCon 10/12, saves per 3, conceded per 2) are not published and were derived by
boundary search over finished gameweeks. This re-runs that check against the
latest results - if FPL changes a rule mid-season, it shows up here rather than
silently skewing every projection.

In [ ]:
report = validate_against_api(FPLClient())
print("rules consistent with live data:", report["ok"])
for name, finding in report["findings"].items():
    print(f"  {name:<20} {finding}")

## 3. Run the pipeline

Set `odds_source="none"` to force the fallback model (useful for comparing what
the market is actually adding). With an API key in `.env`, fixtures with usable
odds are priced from the market and the rest fall back automatically.

In [ ]:
res = run(event=None)          # event=None -> next gameweek; "auto" price source

print(f"\nGameweek {res.event}")
print(f"Market share of total xP: {100*res.exchange_share():.1f}%")
res.provenance

### Fixture view

`xG_home`/`xG_away` are the fitted Dixon-Coles goal rates. Clean-sheet and
goals-conceded expectations are read off the same joint distribution, so they
cannot contradict the match odds.

In [ ]:
res.fixture_table()

## 4. Minutes - the part worth your judgement

The auto model reads FPL availability flags and recent starts. It cannot hear a
press conference, so review the players below and override anything you disagree
with in **`config/minutes_overrides.yaml`**, then re-run section 3.

Shorthand is just expected minutes:

```yaml
players:
  Haaland: 0      # ruled out
  Saka: 60        # expected to be managed
```

In [ ]:
m = res.minutes
flagged = m[(m.status != "a") & (m.price >= 4.5)].sort_values("price", ascending=False)
print("FLAGGED PLAYERS (auto-downgraded):")
flagged[["web_name","team","position","price","status","chance_of_playing",
         "p_start","p_60","xmins","news"]].head(20)

In [ ]:
# Rotation risk: players the model is unsure about who still make the squad shortlist
risky = res.players.merge(m[["player_id","p_start","status"]], on="player_id", suffixes=("","_m"))
risky = risky[(risky.xp > 2.0) & (risky.p_start_m.between(0.25, 0.75))]
print("ROTATION RISKS among viable picks - worth a manual call:")
risky.sort_values("xp", ascending=False)[
    ["web_name","team","position","price","opponent","p_start_m","xmins","xp"]].head(15)

In [ ]:
# Did every override actually apply? A typo here otherwise does nothing silently.
unmatched = [w for w in res.warnings if "override not matched" in w]
print("unmatched overrides:", unmatched or "none - all applied")
overridden = m[m.minutes_source == "override"]
print(f"{len(overridden)} manual override(s) in effect")
overridden[["web_name","team","position","p_start","p_60","xmins"]]

## 5. Expected points

`xp` is the sum of the component columns. Reading the components is usually more
informative than the total - it shows *why* a player rates, and therefore which
assumption to challenge.

In [ ]:
cols = ["web_name","team","position","price","opponent","is_home"] + COMPONENTS + ["xp"]
res.players.head(30)[cols]

In [ ]:
# Best value per million - useful for filling the cheap end of the squad
v = res.players[res.players.xp > 1.0].copy()
v["xp_per_m"] = v.xp / v.price
v.sort_values("xp_per_m", ascending=False).head(20)[
    ["web_name","team","position","price","opponent","xp","xp_per_m"]]

In [ ]:
# Defensive contribution leaders - the newest scoring route, and the one the
# market cannot price directly. FWD DefCon is worth ~0.02 pts/game: ignore it.
d = res.player_fixtures
d[d.xp_defcon > 0].sort_values("xp_defcon", ascending=False).head(15)[
    ["web_name","team","position","price","opponent","dc90","p_defcon","xp_defcon","xp"]]

## 6. Optimise the squad

Exact MILP solve: 15 players, 2/5/5/3, max 3 per club, valid XI, captain doubled,
bench discounted to its autosub value.

On a real Free Hit your budget is **current squad value + bank**, not 100.0 -
set it below.

In [ ]:
BUDGET = 100.0          # <-- your squad value + bank
LOCK   = []             # e.g. ["Haaland"] to force in
BAN    = []             # e.g. ["Saka"] to force out

squad = optimise_free_hit(res.players, budget=BUDGET, locked=LOCK, banned=BAN)
print(squad.summary())

In [ ]:
# Full squad detail with component breakdown
squad.players[["web_name","team","position","price","opponent","is_starter","is_captain"]
              + COMPONENTS + ["xp"]]

### Alternatives

The optimum is often only a fraction of a point clear of quite different squads.
Each alternative differs by at least `diversity` places in the starting XI.

In [ ]:
alts = top_squads(res.players, n=4, budget=BUDGET, locked=LOCK, banned=BAN, diversity=3)
for i, s in enumerate(alts, 1):
    print(f"{i}. {s.formation}  XI xP {s.starting_xp:5.2f}  cost {s.cost:5.1f}m  "
          f"C={s.captain:<14} bench={', '.join(s.bench)}")

In [ ]:
# Captaincy: the armband is worth a full extra return, so check the top few
res.players.head(8)[["web_name","team","position","price","opponent","xp_goals","xp_assists","xp"]]

## 7. How much is the market actually adding?

Run both ways and compare. If the two agree closely, the market is confirming
the model; where they diverge, the market is usually the better estimate - it has
information (team news, money) the statistical model does not.

In [ ]:
res_model = run(odds_source="none", verbose=False)
cmp = res.players[["player_id","web_name","team","position","price","xp"]].merge(
    res_model.players[["player_id","xp"]], on="player_id", suffixes=("_market","_model"))
cmp["diff"] = cmp.xp_market - cmp.xp_model
if cmp["diff"].abs().sum() < 1e-9:
    print("Identical - no market data in use (no API key, or no markets matched).")
else:
    print("Largest disagreements (market vs fallback model):")
    display(cmp.reindex(cmp["diff"].abs().sort_values(ascending=False).index).head(15))

## 8. Export

In [ ]:
from fplfh.config import OUT_DIR, ensure_dirs
ensure_dirs()
res.players.to_csv(OUT_DIR / f"xp_gw{res.event}.csv", index=False)
squad.players.to_csv(OUT_DIR / f"squad_gw{res.event}.csv", index=False)
res.fixture_table().to_csv(OUT_DIR / f"fixtures_gw{res.event}.csv", index=False)
print("written to", OUT_DIR)